In [1]:
# train_models.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, accuracy_score
import pickle
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load dataset (replace with your actual dataset path)
df = pd.read_csv('/workspaces/project_med/notebook/medicine_dataset_2020_2025 (1).csv') 

In [3]:
print("Dataset shape:", df.shape)
print("\nDataset info:")
print(df.info())

Dataset shape: (3000, 23)

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Date                   3000 non-null   str  
 1   Medicine_Name          3000 non-null   str  
 2   Pharmacy_ID            3000 non-null   str  
 3   Pharmacy_Name          3000 non-null   str  
 4   Location               3000 non-null   str  
 5   Address                3000 non-null   str  
 6   Stock_Available        3000 non-null   int64
 7   Units_Sold             3000 non-null   int64
 8   Incoming_Stock         3000 non-null   int64
 9   Season                 3000 non-null   str  
 10  Disease_Trend          3000 non-null   str  
 11  Day_of_Week            3000 non-null   str  
 12  Supplier_Delay         3000 non-null   int64
 13  Demand_Trend           3000 non-null   str  
 14  Prescription_Required  3000 non-null   str  
 15  Price   

In [4]:
# 1. Data Preprocessing
print("\n=== DATA PREPROCESSING ===")

# Convert Date to Day and Month
df['Date'] = pd.to_datetime(df['Date'])
df['Day'] = df['Date'].dt.day
df['Month'] = df['Date'].dt.month
df.drop('Date', axis=1, inplace=True)


=== DATA PREPROCESSING ===


In [5]:
# Drop unnecessary columns
columns_to_drop = ['Pharmacy_Name', 'Address', 'Manufacturer']
df.drop([col for col in columns_to_drop if col in df.columns], axis=1, inplace=True)

In [6]:
# Handle missing values
df.fillna({
    'Stock_Available': df['Stock_Available'].mean(),
    'Units_Sold': df['Units_Sold'].mean(),
    'Incoming_Stock': df['Incoming_Stock'].mean(),
    'Supplier_Delay': df['Supplier_Delay'].mean(),
    'Price': df['Price'].mean(),
    'Restock_Time': df['Restock_Time'].mean()
}, inplace=True)

,Medicine_Name,Pharmacy_ID,Location,Stock_Available,Units_Sold,Incoming_Stock,Season,Disease_Trend,Day_of_Week,Supplier_Delay,...,Prescription_Required,Price,Medicine_Type,Criticality_Level,Supplier_Type,Restock_Time,Dosage_Form,Shortage,Day,Month
0,Paracetamol,P001,Andheri,110,69,44,Flu,Medium,Wednesday,0,...,No,20,Common,Low,Local,2,Tablet,0,1,1
1,Ibuprofen,P002,Dadar,111,49,8,Flu,Medium,Thursday,1,...,No,30,Common,Low,Local,5,Tablet,0,2,1
2,Crocin,P003,Thane,151,71,43,Flu,High,Friday,2,...,No,25,Common,Low,Local,3,Tablet,0,3,1
3,Cetirizine,P004,Bandra,96,77,21,Flu,High,Saturday,0,...,No,15,Common,Low,Regional,5,Syrup,0,4,1
4,Azithromycin,P001,Andheri,100,83,19,Flu,High,Sunday,1,...,Yes,90,Rare,High,Imported,12,Capsule,0,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,Amoxicillin,P004,Bandra,166,14,29,Normal,Low,Tuesday,1,...,Yes,80,Critical,High,National,8,Capsule,0,14,3
2996,Metformin,P001,Andheri,139,13,41,Normal,Low,Wednesday,1,...,Yes,45,Chronic,Medium,National,5,Tablet,0,15,3
2997,Amlodipine,P002,Dadar,118,17,56,Normal,Low,Thursday,0,...,Yes,35,Chronic,Medium,National,8,Tablet,0,16,3
2998,Insulin,P003,Thane,90,81,21,Normal,Low,Friday,0,...,Yes,95,Chronic,High,National,5,Injection,0,17,3


In [7]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", categorical_cols)

Categorical columns: ['Medicine_Name', 'Pharmacy_ID', 'Location', 'Season', 'Disease_Trend', 'Day_of_Week', 'Demand_Trend', 'Prescription_Required', 'Medicine_Type', 'Criticality_Level', 'Supplier_Type', 'Dosage_Form']


In [8]:
# Encode categorical columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

In [9]:
# Convert target to proper types
df['Units_Sold'] = df['Units_Sold'].astype(float)
df['Shortage'] = df['Shortage'].astype(int)

print("Processed dataset shape:", df.shape)
print("\nFeature columns:", df.columns.tolist())

Processed dataset shape: (3000, 21)

Feature columns: ['Medicine_Name', 'Pharmacy_ID', 'Location', 'Stock_Available', 'Units_Sold', 'Incoming_Stock', 'Season', 'Disease_Trend', 'Day_of_Week', 'Supplier_Delay', 'Demand_Trend', 'Prescription_Required', 'Price', 'Medicine_Type', 'Criticality_Level', 'Supplier_Type', 'Restock_Time', 'Dosage_Form', 'Shortage', 'Day', 'Month']


In [10]:
# 2. Train-Test Split
X = df.drop(['Units_Sold', 'Shortage'], axis=1)
y_demand = df['Units_Sold']
y_shortage = df['Shortage']

X_train, X_test, y_demand_train, y_demand_test, y_shortage_train, y_shortage_test = train_test_split(
    X, y_demand, y_shortage, test_size=0.2, random_state=42
)

In [11]:

print("\nTrain set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


Train set shape: (2400, 19)
Test set shape: (600, 19)


In [12]:
# 3A. Demand Prediction Model (RandomForestRegressor)
print("\n=== TRAINING DEMAND MODEL ===")
demand_model = RandomForestRegressor(n_estimators=100, random_state=42)
demand_model.fit(X_train, y_demand_train)


=== TRAINING DEMAND MODEL ===


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [13]:
# Predictions and evaluation
demand_pred_train = demand_model.predict(X_train)
demand_pred_test = demand_model.predict(X_test)
demand_mae_train = mean_absolute_error(y_demand_train, demand_pred_train)
demand_mae_test = mean_absolute_error(y_demand_test, demand_pred_test)

print(f"Demand Model - Train MAE: {demand_mae_train:.2f}")
print(f"Demand Model - Test MAE: {demand_mae_test:.2f}")

Demand Model - Train MAE: 2.56
Demand Model - Test MAE: 6.58


In [14]:
# 3B. Shortage Prediction Model (RandomForestClassifier)
print("\n=== TRAINING SHORTAGE MODEL ===")
shortage_model = RandomForestClassifier(n_estimators=100, random_state=42)
shortage_model.fit(X_train, y_shortage_train)


=== TRAINING SHORTAGE MODEL ===


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [15]:
# Predictions and evaluation
shortage_pred_train = shortage_model.predict(X_train)
shortage_pred_test = shortage_model.predict(X_test)
shortage_acc_train = accuracy_score(y_shortage_train, shortage_pred_train)
shortage_acc_test = accuracy_score(y_shortage_test, shortage_pred_test)

print(f"Shortage Model - Train Accuracy: {shortage_acc_train:.3f}")
print(f"Shortage Model - Test Accuracy: {shortage_acc_test:.3f}")

Shortage Model - Train Accuracy: 1.000
Shortage Model - Test Accuracy: 0.998


In [16]:
# 4. Save models and encoders
print("\n=== SAVING MODELS & ENCODERS ===")
joblib.dump(demand_model, 'demand_model.pkl')
joblib.dump(shortage_model, 'shortage_model.pkl')
joblib.dump(label_encoders, 'encoders.pkl')
joblib.dump(X.columns.tolist(), 'feature_names.pkl')

print("Models and encoders saved successfully!")
print("\nFeature names saved:", X.columns.tolist())


=== SAVING MODELS & ENCODERS ===
Models and encoders saved successfully!

Feature names saved: ['Medicine_Name', 'Pharmacy_ID', 'Location', 'Stock_Available', 'Incoming_Stock', 'Season', 'Disease_Trend', 'Day_of_Week', 'Supplier_Delay', 'Demand_Trend', 'Prescription_Required', 'Price', 'Medicine_Type', 'Criticality_Level', 'Supplier_Type', 'Restock_Time', 'Dosage_Form', 'Day', 'Month']
